# 03 · Dim_Cliente
**IDY1102 · EP3 · DataRetail 360**

| Campo | Detalle |
|---|---|
| Responsable | [Mara Cantillana] |
| Tabla destino | `dim_cliente` |
| SCD | **Tipo 2**: comuna de residencia, segmento · Tipo 1: tramo etario, fecha de primer registro, `geografia_key` |
| Clave natural | `rut_anonimizado` (hash del RUT con sal; el RUT nunca se guarda) |
| Depende de | `dim_geografia` (notebook 01) y de la variable `RUT_SALT` en `.env` |
| Fuentes | `clientes_snapshot_1.csv` y `clientes_snapshot_2.csv` |
| Alimenta a | `fact_ventas` (notebook 04) |

> **Cómo usar esta plantilla.** Las celdas marcadas **✏️ TODO** son la parte que debe implementar el integrante
> responsable (las reglas de calidad). Todo lo demás ya está conectado a los módulos compartidos de `src/`
> (carga idempotente, SCD, auditoría, logging). Antes de commitear: *Restart & Run All* debe terminar sin errores.

> ⚠️ **Datos personales:** este notebook maneja RUT, nombre y correo. **No los imprimas** (`head()`, `print`) ni los dejes en las salidas guardadas del notebook: el `.ipynb` se sube a Git. Muestra solo columnas anonimizadas o conteos.

## 0. Setup

In [1]:
import os, sys
from datetime import date
from pathlib import Path

import pandas as pd

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

from src import db, extract, medallion, quality as q, schema
from src.config import get_settings
from src.logger import get_logger

PROCESO = "dim_cliente"
log = get_logger(PROCESO)
cfg = get_settings()
engine = db.get_engine(cfg)
schema.crear_esquema(engine)                  # idempotente: crea solo lo que falta
schema.sembrar_miembros_desconocidos(engine)  # miembro -1 "No informado" en cada dimensión
pd.set_option("display.width", 160)

2026-10-25 19:56:41,376 | INFO     | datarretail360.db | Conectado a sqlserver (DataRetail360_DW)


## 1. Parámetros

In [2]:
# Snapshot a procesar: lo define run_pipeline.py (variable de entorno SNAPSHOT) o puedes cambiarlo a mano.
SNAPSHOT = int(os.getenv("SNAPSHOT", "1"))
FECHAS_EFECTIVAS = {1: date(2025, 1, 1), 2: date(2025, 6, 1)}   # desde cuándo rige cada snapshot
FECHA_EFECTIVA = FECHAS_EFECTIVAS[SNAPSHOT]
ARCHIVO = cfg.data_dir / f"clientes_snapshot_{SNAPSHOT}.csv"
print(f"Snapshot {SNAPSHOT} · fecha efectiva {FECHA_EFECTIVA} · {ARCHIVO.name}")

Snapshot 1 · fecha efectiva 2025-01-01 · clientes_snapshot_1.csv


## 2. Extract
Lee la fuente cruda y la guarda **sin cambios** en la capa Bronze.

In [3]:
def extraer(ruta):
    """Lee la fuente cruda y la deja intacta en la capa Bronze (contiene datos personales: la carpeta data/ no se versiona)."""
    df = extract.leer_csv(ruta)
    medallion.guardar_capa(df, "bronze", f"clientes_snapshot_{SNAPSHOT}")
    print("Columnas:", list(df.columns), "· filas:", len(df))   # sin mostrar datos personales
    return df

## 3. Transform  ✏️ TODO
**Implementa al menos 3 reglas de calidad** (la rúbrica del EP3 y la Actividad 2.2 lo piden). Estas son las que traen los datos de ejemplo:

| Problema en los datos de ejemplo | Dimensión de calidad | Pista |
|---|---|---|
| RUT con dígito verificador incorrecto o texto que no es un RUT (2 filas) | Validez | `q.validar_ruts` |
| Mismo cliente repetido con RUT escrito distinto (con/sin puntos), correo y nombre distintos (2 filas) | Unicidad | `q.anonimizar_serie` normaliza el RUT: luego `duplicated` sobre `rut_anonimizado` |
| Comuna vacía (1 fila) o código que no existe en `dim_geografia` (1 fila) | Completitud / Consistencia | no se rechaza: va al miembro `-1`. Usa `db.mapa_claves` |
| Fecha de nacimiento, nombre y correo no deben llegar al DW | Minimización (Ley 21.719) | `q.tramo_etario` y no incluir esas columnas en la salida |

> Con los datos de ejemplo el snapshot 1 debería dejar fuera 4 filas (2 RUT inválidos y 2 duplicados) como conteo de control.

In [4]:
# Contrato de salida de transformar(): columnas que espera la carga (no lo borres)
from src.config import exigir_rut_salt
COLUMNAS_SALIDA = ["rut_anonimizado", "tramo_etario", "segmento", "codigo_comuna", "geografia_key", "fecha_primer_registro"]

In [5]:
from src.config import exigir_rut_salt

def transformar(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Limpia, valida y anonimiza clientes cumpliendo Ley 21.719."""
    rechazos_lista = []
    
    # 1. Validez: Separar RUTs inválidos
    mascara_rut_valido = df["rut"].apply(q.rut_valido)
    df_valido, r_rut = q.separar_rechazos(
        df, ~mascara_rut_valido, "rut_invalido", detalle="RUT con formato o DV invalido"
    )
    rechazos_lista.append(r_rut)
    
    # 2. Unicidad y Minimización: Anonimizar con sal y deduplicar
    salt = exigir_rut_salt()
    df_valido = df_valido.copy()
    df_valido["rut_anonimizado"] = q.anonimizar_serie(df_valido["rut"], salt)
    
    mascara_dup = df_valido.duplicated(subset=["rut_anonimizado"], keep="first")
    df_dedup, r_dup = q.separar_rechazos(
        df_valido, mascara_dup, "cliente_duplicado", detalle="RUT duplicado en snapshot"
    )
    rechazos_lista.append(r_dup)
    
    # 3. Minimización y Fechas (Ley 21.719)
    df_limpio = df_dedup.copy()
    nacimiento_dt = q.parsear_fechas(df_limpio["fecha_nacimiento"])
    df_limpio["tramo_etario"] = q.tramo_etario(nacimiento_dt, FECHA_EFECTIVA)
    df_limpio["fecha_primer_registro"] = q.parsear_fechas(df_limpio["fecha_registro"]).dt.date
    
    # 4. Consistencia / Completitud: Mapear geografia_key desde codigo_comuna
    mapa_geo = db.mapa_claves(engine, "dim_geografia", "codigo_comuna", "geografia_key")
    df_limpio["codigo_comuna"] = df_limpio["codigo_comuna"].astype(str).str.strip()
    df_limpio["geografia_key"] = df_limpio["codigo_comuna"].map(mapa_geo).fillna(-1).astype(int)
    df_limpio.loc[df_limpio["geografia_key"] == -1, "codigo_comuna"] = "-1"
    
    # 5. Normalizar textos
    df_limpio["segmento"] = q.normalizar_texto(df_limpio["segmento"])
    
    # Consolidar rechazos desempaquetando los DataFrames (*)
    df_rechazos = q.consolidar_rechazos(*rechazos_lista)
    
    return df_limpio[COLUMNAS_SALIDA], df_rechazos

## 4. Load
Carga idempotente con el helper compartido y ejecución completa con auditoría (`Ejecucion`).

In [6]:
ATRIBUTOS_SCD2 = ["codigo_comuna", "segmento"]   # si el cliente se cambia de comuna o de segmento, se versiona
ATRIBUTOS_SCD1 = ["tramo_etario", "fecha_primer_registro", "geografia_key"]

def cargar(df_limpio: pd.DataFrame, df_rechazos: pd.DataFrame) -> dict:
    """Registra los rechazos y carga la dimensión aplicando SCD Tipo 2."""
    assert set(COLUMNAS_SALIDA) <= set(df_limpio.columns), "transformar() no devolvió las columnas esperadas"
    db.registrar_rechazos(engine, PROCESO, df_rechazos)
    return db.aplicar_scd2(engine, "dim_cliente", df_limpio, clave_natural="rut_anonimizado",
                           atributos_scd2=ATRIBUTOS_SCD2, fecha_efectiva=FECHA_EFECTIVA,
                           key_col="cliente_key", atributos_scd1=ATRIBUTOS_SCD1)

In [7]:
with db.Ejecucion(engine, PROCESO) as run:
    df_crudo = extraer(ARCHIVO)
    run.leidas = len(df_crudo)
    df_limpio, df_rechazos = transformar(df_crudo)
    medallion.guardar_capa(df_limpio, "silver", f"clientes_snapshot_{SNAPSHOT}")
    resumen = cargar(df_limpio, df_rechazos)
    run.cargadas = resumen["nuevos"] + resumen["versiones_nuevas"]
    run.rechazadas = len(df_rechazos)
resumen

2026-10-25 19:57:12,071 | INFO     | datarretail360.db | [dim_cliente] INICIO
2026-10-25 19:57:12,146 | INFO     | datarretail360.extract | Extraídas 30 filas desde clientes_snapshot_1.csv
2026-10-25 19:57:12,354 | INFO     | datarretail360.medallion | Capa bronze: 30 filas guardadas en clientes_snapshot_1.parquet
Columnas: ['rut', 'nombre', 'email', 'codigo_comuna', 'segmento', 'fecha_registro', 'fecha_nacimiento', 'origen_archivo', 'fila_origen'] · filas: 30
2026-10-25 19:57:12,657 | INFO     | datarretail360.medallion | Capa silver: 26 filas guardadas en clientes_snapshot_1.parquet
2026-10-25 19:57:12,781 | INFO     | datarretail360.db | Carga auditoria_rechazos: {'insertadas': 4, 'ya_existian': 0, 'duplicadas_en_lote': 0}
2026-10-25 19:57:12,782 | WARNING  | datarretail360.db | [dim_cliente] 4 filas rechazadas (nuevas en auditoría: 4)
2026-10-25 19:57:12,846 | INFO     | datarretail360.db | SCD2 dim_cliente (efectiva 2025-01-01): {'nuevos': 26, 'versiones_nuevas': 0, 'actualizados_

{'nuevos': 26,
 'versiones_nuevas': 0,
 'actualizados_scd1': 0,
 'sin_cambio': 0,
 'omitidos': 0}

## 5. Validación
Chequeos automáticos sobre lo ya cargado. Si algo está mal, el notebook **falla** (no sigue en silencio).

In [8]:
def validar():
    dim = db.leer_tabla(engine, "dim_cliente")
    problemas = q.chequear_scd2(dim, "rut_anonimizado", "cliente_key")
    assert not problemas, problemas
    assert (dim["cliente_key"] == -1).sum() == 1, "debe existir exactamente un miembro 'No informado' (-1)"
    log.info("Validación de dim_cliente OK (%s filas)", len(dim))

validar()

2026-10-25 19:57:22,516 | INFO     | datarretail360.dim_cliente | Validación de dim_cliente OK (27 filas)


In [9]:
# Idempotencia: volver a cargar los MISMOS datos no debe cambiar nada
filas_antes = len(db.leer_tabla(engine, "dim_cliente"))
resumen_2 = cargar(df_limpio, pd.DataFrame(columns=q.COLUMNAS_RECHAZO))
assert len(db.leer_tabla(engine, "dim_cliente")) == filas_antes, "la carga no es idempotente"
print("Idempotencia OK:", resumen_2)

2026-10-25 19:57:26,290 | INFO     | datarretail360.db | SCD2 dim_cliente (efectiva 2025-01-01): {'nuevos': 0, 'versiones_nuevas': 0, 'actualizados_scd1': 0, 'sin_cambio': 26, 'omitidos': 0}
Idempotencia OK: {'nuevos': 0, 'versiones_nuevas': 0, 'actualizados_scd1': 0, 'sin_cambio': 26, 'omitidos': 0}


## 6. Evidencia
Estas salidas son las que se capturan para el informe (apartados 6, 7 y 9 de la pauta EP3).

In [10]:
def conteos_por_capa() -> pd.DataFrame:
    """Filas en cada capa Medallion para este snapshot: trazabilidad fuente → bronze → silver → gold."""
    bronze = medallion.leer_capa("bronze", f"clientes_snapshot_{SNAPSHOT}")
    silver = medallion.leer_capa("silver", f"clientes_snapshot_{SNAPSHOT}")
    gold = db.leer_tabla(engine, "dim_cliente", solo_vigentes=True)
    gold = gold[gold["cliente_key"] != -1]
    return pd.DataFrame({"capa": ["bronze (crudo)", "silver (limpio)", "gold (vigentes en el DW)"],
                          "filas": [len(bronze), len(silver), len(gold)]})

conteos_por_capa()

,capa,filas
0,bronze (crudo),30
1,silver (limpio),26
2,gold (vigentes en el DW),26


In [ ]:
# Historia SCD2 de un registro de ejemplo (un cliente que se cambia de comuna en el snapshot 2): una fila por versión
dim = db.leer_tabla(engine, "dim_cliente")
versiones = dim.groupby("rut_anonimizado").size()
ejemplos = versiones[versiones > 1].index[:1]
dim[dim["rut_anonimizado"].isin(ejemplos)].sort_values(["rut_anonimizado", "valido_desde"])

In [ ]:
# Rechazos de este proceso y últimas ejecuciones registradas (auditoría)
rech = db.leer_tabla(engine, "auditoria_rechazos")
display(rech[rech["proceso"] == PROCESO].groupby("motivo_rechazo").size().rename("filas").to_frame())
ejec = db.leer_tabla(engine, "auditoria_ejecuciones")
ejec[ejec["proceso"] == PROCESO].tail(4)[["inicio", "estado", "filas_leidas", "filas_cargadas", "filas_rechazadas"]]

## 7. Decisiones de diseño

### 1. Justificación de SCD Tipo 2 vs. Tipo 3 para comuna y segmento
- **SCD Tipo 2 obligatorio:** El dolor principal del caso DataRetail 360 es la distorsión de cifras históricas. Si un cliente cambia de comuna de residencia o segmento comercial, aplicar Tipo 1 sobrescribiría su historial, reasignando retroactivamente sus compras pasadas a una nueva ubicación geográfica. Tipo 3 solo conserva el valor anterior en una columna adicional, perdiendo la trazabilidad si el cliente se muda más de una vez y vulnerando el principio de responsabilidad y auditabilidad exigido por la **Ley 21.719**. Dado que la tasa de mudanza de clientes es baja, el costo marginal de almacenamiento es mínimo frente al beneficio analítico.

### 2. Atributo `geografia_key` como SCD Tipo 1
- `codigo_comuna` actúa como el disparador del versionamiento histórico cuando el cliente se muda de comuna. Mantener `geografia_key` como SCD1 permite corregir re-mapeos técnicos del catálogo geográfico sin generar versiones artificiales del cliente si su comuna real no ha variado.

### 3. Privacidad y tratamiento de datos en Capa Bronze
- La capa Bronze almacena los datos crudos originales (incluyendo el RUT para fines de ingesta inmutable y auditoría interna). La privacidad legal se resguarda aislando la carpeta `data/` (excluida del control de versiones en `.gitignore`) y garantizando que a partir de las capas Silver y Gold únicamente viaje el identificador seudonimizado `rut_anonimizado` (SHA-256 + `RUT_SALT`), excluyendo nombres, correos y fechas exactas de nacimiento.